# SXO Page-Query Dynamics

Understand how top pages, rankings, and queries shift in a **single recent window** with an SXO lens.

## What this notebook helps answer
- Which top pages moved most in the last `X` weeks?
- Which queries likely drove those page-level changes?
- Do those changes look more like demand shifts, discoverability shifts, or SERP competitiveness shifts?

## How to use
1. Run **Setup (run once)**.
2. Set values in **Parameters** (especially `RECENT_WEEKS` and `EXCLUDED_PAGE_PATHS`).
3. Run all cells from top to bottom.
4. Use the interpretation/checklist at the end to decide action.

This notebook focuses on practical diagnosis, not traffic reporting.

In [ ]:
#@title Setup (run once)
import os
import sys
from datetime import date, timedelta

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()
    if not os.path.exists("lla-data"):
        !git clone -q https://github.com/aidanm-lla/lla-data.git
    repo = os.path.abspath("lla-data")
    if repo not in sys.path:
        sys.path.insert(0, repo)
    !pip install -U -q "plotly>=6.1.1" "kaleido>=1.2.0"
else:
    for p in ("..", "../.."):
        ap = os.path.abspath(p)
        if ap not in sys.path:
            sys.path.insert(0, ap)

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from google.cloud import bigquery

import lifeline_theme
from lla_data import config
from lla_data.bq import build_date_params, get_client, load_sql_template, run_query

lifeline_theme.inject_fonts()
client = get_client()

In [ ]:
#@title Parameters
TOP_N_PAGES = 25 #@param {type:"integer"}
RECENT_WEEKS = 8 #@param {type:"integer"}
EXCLUDED_PAGE_PATHS = "/, /get-help/national-services/national-alcohol-other-drug-hotline" #@param {type:"string"}
MIN_IMPRESSIONS_WEEKLY = 0 #@param {type:"integer"}
MIN_QUERY_IMPRESSIONS = 0 #@param {type:"integer"}
FOCUS_PAGE_COUNT = 8 #@param {type:"integer"}

if RECENT_WEEKS < 2:
    raise ValueError("RECENT_WEEKS must be at least 2 so comparisons are meaningful.")

# Accept comma-separated paths, with optional newlines if you edit the cell manually.
excluded_page_paths = sorted(
    {
        path.strip()
        for path in EXCLUDED_PAGE_PATHS.replace("\n", ",").split(",")
        if path.strip()
    }
)

recent_days = RECENT_WEEKS * 7
comparison_days = recent_days * 2
analysis_end_date = date.today() - timedelta(days=2)

# Pull only what we need: recent window + prior comparison window.
analysis_start_date = analysis_end_date - timedelta(days=comparison_days - 1)

print(f"Project: {config.PROJECT_ID}")
print(f"Search Console dataset: {config.SEARCHCONSOLE_DATASET}")
print(f"Analysis window: {analysis_start_date} to {analysis_end_date} ({comparison_days} days)")
print(f"Recent focus window: last {RECENT_WEEKS} week(s) ({recent_days} days)")
print(f"Excluded page paths: {excluded_page_paths or '[none]'}")

In [ ]:
# Freshness / sanity check
freshness_sql = f"""
SELECT
  (SELECT MAX(report_date) FROM `{config.PROJECT_ID}.{config.SEARCHCONSOLE_DATASET}.seo_page_daily`) AS max_page_date,
  (SELECT MAX(report_date) FROM `{config.PROJECT_ID}.{config.SEARCHCONSOLE_DATASET}.curated_search_query_page_daily`) AS max_query_date
"""

freshness = run_query(client, freshness_sql)
max_page_date = pd.to_datetime(freshness.loc[0, "max_page_date"]).date()
max_query_date = pd.to_datetime(freshness.loc[0, "max_query_date"]).date()
effective_end_date = min(analysis_end_date, max_page_date, max_query_date)

print(f"Latest page-level date: {max_page_date}")
print(f"Latest query-level date: {max_query_date}")
print(f"Effective analysis end date: {effective_end_date}")

if effective_end_date <= analysis_start_date:
    raise ValueError("No usable data in configured window. Adjust dates or check data freshness.")

availability_sql = f"""
WITH page_weekly AS (
  SELECT
    DATE_TRUNC(report_date, WEEK(MONDAY)) AS week_start,
    SUM(gsc_clicks) AS clicks
  FROM `{config.PROJECT_ID}.{config.SEARCHCONSOLE_DATASET}.seo_page_daily`
  WHERE report_date BETWEEN DATE(@start_date) AND DATE(@end_date)
  GROUP BY week_start
),
query_weekly AS (
  SELECT
    DATE_TRUNC(report_date, WEEK(MONDAY)) AS week_start,
    SUM(clicks) AS clicks
  FROM `{config.PROJECT_ID}.{config.SEARCHCONSOLE_DATASET}.curated_search_query_page_daily`
  WHERE report_date BETWEEN DATE(@start_date) AND DATE(@end_date)
  GROUP BY week_start
)
SELECT
  (SELECT COUNTIF(clicks > 0) FROM page_weekly) AS page_nonzero_weeks,
  (SELECT COUNTIF(clicks > 0) FROM query_weekly) AS query_nonzero_weeks
"""

availability = run_query(
    client,
    availability_sql,
    params=[
        bigquery.ScalarQueryParameter("start_date", "DATE", analysis_start_date),
        bigquery.ScalarQueryParameter("end_date", "DATE", effective_end_date),
    ],
)

page_nonzero_weeks = int(availability.loc[0, "page_nonzero_weeks"])
query_nonzero_weeks = int(availability.loc[0, "query_nonzero_weeks"])

print(f"Weeks with non-zero page clicks in analysis window: {page_nonzero_weeks}")
print(f"Weeks with non-zero query clicks in analysis window: {query_nonzero_weeks}")

if page_nonzero_weeks < RECENT_WEEKS or query_nonzero_weeks < RECENT_WEEKS:
    print(
        "Warning: fewer non-zero GSC weeks are available than RECENT_WEEKS. "
        "Leading weeks may appear as zero until data backfill catches up."
    )

In [ ]:
# Load weekly page dynamics for top pages
page_sql = load_sql_template(
    "sql/notebooks/seo_sxo_page_weekly_dynamics.sql",
    project_id=config.PROJECT_ID,
    searchconsole_dataset=config.SEARCHCONSOLE_DATASET,
)

page_params = [
    bigquery.ScalarQueryParameter("start_date", "DATE", analysis_start_date),
    bigquery.ScalarQueryParameter("end_date", "DATE", effective_end_date),
    bigquery.ScalarQueryParameter("top_n_pages", "INT64", TOP_N_PAGES),
    bigquery.ArrayQueryParameter("excluded_page_paths", "STRING", excluded_page_paths),
    bigquery.ScalarQueryParameter("min_impressions_weekly", "INT64", MIN_IMPRESSIONS_WEEKLY),
]

df_page_weekly = run_query(client, page_sql, params=page_params)
if df_page_weekly.empty:
    raise ValueError("No weekly page data returned. Try lowering thresholds or widening date windows.")

df_page_weekly["week_start"] = pd.to_datetime(df_page_weekly["week_start"])

print(f"Rows returned: {len(df_page_weekly):,}")
df_page_weekly.head()

In [ ]:
# Derive recent movers from weekly data
latest_week = df_page_weekly["week_start"].max()
recent_week_cutoff = latest_week - pd.Timedelta(days=(RECENT_WEEKS - 1) * 7)
prior_week_cutoff = recent_week_cutoff - pd.Timedelta(days=RECENT_WEEKS * 7)

recent = df_page_weekly[df_page_weekly["week_start"] >= recent_week_cutoff].copy()
prior = df_page_weekly[
    (df_page_weekly["week_start"] >= prior_week_cutoff)
    & (df_page_weekly["week_start"] < recent_week_cutoff)
].copy()

recent_summary = (
    recent.groupby("page_path", as_index=False)
    .agg(
        recent_clicks=("clicks", "sum"),
        recent_impressions=("impressions", "sum"),
    )
)
recent_summary["recent_ctr"] = recent_summary["recent_clicks"] / recent_summary["recent_impressions"].replace(0, pd.NA)

prior_summary = (
    prior.groupby("page_path", as_index=False)
    .agg(
        prior_clicks=("clicks", "sum"),
        prior_impressions=("impressions", "sum"),
    )
)
prior_summary["prior_ctr"] = prior_summary["prior_clicks"] / prior_summary["prior_impressions"].replace(0, pd.NA)

recent["position_x_impressions"] = recent["avg_position"] * recent["impressions"]
prior["position_x_impressions"] = prior["avg_position"] * prior["impressions"]

recent_position = (
    recent.groupby("page_path", as_index=False)
    .agg(
        recent_position_weighted=("position_x_impressions", "sum"),
        recent_impressions_for_position=("impressions", "sum"),
    )
)
recent_position["recent_avg_position"] = (
    recent_position["recent_position_weighted"]
    / recent_position["recent_impressions_for_position"].replace(0, pd.NA)
)
recent_position = recent_position[["page_path", "recent_avg_position"]]

prior_position = (
    prior.groupby("page_path", as_index=False)
    .agg(
        prior_position_weighted=("position_x_impressions", "sum"),
        prior_impressions_for_position=("impressions", "sum"),
    )
)
prior_position["prior_avg_position"] = (
    prior_position["prior_position_weighted"]
    / prior_position["prior_impressions_for_position"].replace(0, pd.NA)
)
prior_position = prior_position[["page_path", "prior_avg_position"]]

page_movers = recent_summary.merge(prior_summary, on="page_path", how="left")
page_movers = page_movers.merge(recent_position, on="page_path", how="left")
page_movers = page_movers.merge(prior_position, on="page_path", how="left")

for col in ["prior_clicks", "prior_impressions", "prior_ctr", "prior_avg_position"]:
    page_movers[col] = page_movers[col].fillna(0)

page_movers["click_delta"] = page_movers["recent_clicks"] - page_movers["prior_clicks"]
page_movers["impression_delta"] = page_movers["recent_impressions"] - page_movers["prior_impressions"]
page_movers["ctr_delta"] = page_movers["recent_ctr"] - page_movers["prior_ctr"]
page_movers["position_delta"] = page_movers["recent_avg_position"] - page_movers["prior_avg_position"]

page_movers = page_movers.sort_values("click_delta", ascending=False)
page_movers.head(15)

## Chart 1: Weekly clicks for top movers

This chart shows weekly click trend lines for the pages with the strongest recent momentum.

Why this helps:
- Quickly spot whether growth/decline is broad, sudden, or concentrated in a few pages.
- Separate stable trend movement from one-off spikes before taking action.

How to read it:
- Each line is one page.
- X-axis is week start date.
- The chart title shows the exact date range.
- Flat zero points can appear when a page had no measurable impressions/clicks in that week.

In [ ]:
# Weekly trends for pages with strongest recent momentum
focus_pages = page_movers.head(FOCUS_PAGE_COUNT)["page_path"].tolist()

df_focus = df_page_weekly[
    (df_page_weekly["page_path"].isin(focus_pages))
    & (df_page_weekly["week_start"] >= recent_week_cutoff)
    & (df_page_weekly["week_start"] <= latest_week)
].copy()

# Ensure the chart always shows the full recent-week window, even when
# a page has no row in a given week (for example, filtered by low impressions).
recent_weeks = pd.date_range(
    start=recent_week_cutoff.normalize(),
    end=latest_week.normalize(),
    freq="W-MON",
)

focus_grid = pd.MultiIndex.from_product(
    [focus_pages, recent_weeks],
    names=["page_path", "week_start"],
).to_frame(index=False)

df_focus = focus_grid.merge(
    df_focus[["page_path", "week_start", "clicks"]],
    on=["page_path", "week_start"],
    how="left",
)

df_focus["clicks"] = df_focus["clicks"].fillna(0)
df_focus = df_focus.sort_values(["page_path", "week_start"])

# Format the title like a simple page heading hierarchy.
date_label = f"{recent_weeks.min().strftime('%d %b %Y')} to {recent_weeks.max().strftime('%d %b %Y')}"
chart_title = (
    "Weekly clicks from search"
    f"<br><span style='font-size:0.8em; font-weight:400;'>{date_label}</span>"
)
if excluded_page_paths:
    exclude_label = ", ".join(excluded_page_paths[:3])
    if len(excluded_page_paths) > 3:
        exclude_label = f"{exclude_label}, +{len(excluded_page_paths) - 3} more"
    chart_title += (
        f"<br><span style='font-size:0.55em; font-weight:400;'>Excluded: {exclude_label}</span>"
    )

fig_trend = px.line(
    df_focus,
    x="week_start",
    y="clicks",
    color="page_path",
    markers=True,
    template="lifeline",
    title=chart_title,
    labels={"page_path": "Page path", "clicks": "Clicks", "week_start": "Week"},
)
fig_trend.update_xaxes(range=[recent_weeks.min(), recent_weeks.max()])
lifeline_theme.add_lifeline_logo(fig_trend)
fig_trend.show()

print(
    f"Trend chart coverage: {recent_weeks.min().date()} to {recent_weeks.max().date()} "
    f"({len(recent_weeks)} weeks)"
)

## Chart 2: Ranking vs CTR snapshot

This scatterplot compares ranking quality and click-through performance in the recent window.

Why this helps:
- Find pages with decent rankings but weak CTR (snippet/message opportunity).
- Find pages with strong CTR but weaker rankings (SEO discoverability opportunity).

How to read it:
- Further left = better average position.
- Higher = better CTR.
- Bigger bubbles = higher impressions.
- Color shows click change vs prior window (`click_delta`).

In [ ]:
# Ranking vs demand snapshot (recent window)
fig_scatter = px.scatter(
    page_movers,
    x="recent_avg_position",
    y="recent_ctr",
    size="recent_impressions",
    color="click_delta",
    hover_data=["page_path", "recent_clicks", "prior_clicks", "impression_delta", "position_delta"],
    template="lifeline",
    title="Recent page performance: ranking vs CTR, sized by impressions",
)
fig_scatter.update_xaxes(autorange="reversed", title="Average position (lower is better)")
fig_scatter.update_yaxes(title="CTR")
lifeline_theme.add_lifeline_logo(fig_scatter)
fig_scatter.show()

## Table 1: Rising and falling pages

These tables list the biggest winners and losers by click change.

Why this helps:
- Prioritize which pages need investigation first.
- Compare whether movement comes from clicks, impressions, ranking, or CTR.

Tip: Start with the largest negative `click_delta` rows to triage risk quickly.

In [ ]:
# Recent movers table (positive and negative)
cols = [
    "page_path",
    "recent_clicks",
    "prior_clicks",
    "click_delta",
    "recent_impressions",
    "prior_impressions",
    "impression_delta",
    "recent_avg_position",
    "prior_avg_position",
    "position_delta",
    "recent_ctr",
    "prior_ctr",
    "ctr_delta",
]

print("Top rising pages by click delta")
display(page_movers[cols].head(15))

print("Top falling pages by click delta")
display(page_movers[cols].sort_values("click_delta").head(15))

## Query-level drilldown input

Next we load query-level changes for the same analysis window, so we can explain *why* page performance moved.

This bridges page-level symptoms (what changed) to query-level causes (what drove it).

In [ ]:
# Load query-level recent changes for top pages
query_sql = load_sql_template(
    "sql/notebooks/seo_sxo_query_page_recent_change.sql",
    project_id=config.PROJECT_ID,
    searchconsole_dataset=config.SEARCHCONSOLE_DATASET,
)

query_params = [
    bigquery.ScalarQueryParameter("start_date", "DATE", analysis_start_date),
    bigquery.ScalarQueryParameter("end_date", "DATE", effective_end_date),
    bigquery.ScalarQueryParameter("recent_days", "INT64", recent_days),
    bigquery.ScalarQueryParameter("top_n_pages", "INT64", TOP_N_PAGES),
    bigquery.ArrayQueryParameter("excluded_page_paths", "STRING", excluded_page_paths),
    bigquery.ScalarQueryParameter("min_query_impressions", "INT64", MIN_QUERY_IMPRESSIONS),
]

df_query_changes = run_query(client, query_sql, params=query_params)
if df_query_changes.empty:
    raise ValueError("No query-change rows returned. Try lowering MIN_QUERY_IMPRESSIONS.")

print(f"Rows returned: {len(df_query_changes):,}")
df_query_changes.head()

## Table 2: Top rising and falling queries

These tables show the strongest positive and negative query movements across your top pages.

Why this helps:
- Identify specific intents/themes driving page gains or losses.
- Distinguish new/lost queries from changes in existing queries.

In [ ]:
# Query momentum diagnostics
rising_queries = df_query_changes.sort_values("click_delta", ascending=False).head(30)
falling_queries = df_query_changes.sort_values("click_delta", ascending=True).head(30)

# Uncomment below for detailed info on what queries are rising and falling

# print("Top rising queries across top pages")
# display(rising_queries[[
#     "page_path",
#     "query",
#     "recent_clicks",
#     "prior_clicks",
#     "click_delta",
#     "recent_impressions",
#     "prior_impressions",
#     "impression_delta",
#     "recent_avg_position",
#     "prior_avg_position",
#     "avg_position_delta",
# ]].reset_index(drop=True))

# print("Top falling queries across top pages")
# display(falling_queries[[
#     "page_path",
#     "query",
#     "recent_clicks",
#     "prior_clicks",
#     "click_delta",
#     "recent_impressions",
#     "prior_impressions",
#     "impression_delta",
#     "recent_avg_position",
#     "prior_avg_position",
#     "avg_position_delta",
# ]].reset_index(drop=True))

## Chart 3: Query contribution waterfall

This waterfall explains *why* a page moved by showing which queries added to or dragged down its net click change.

Why this helps:
- It turns a page-level result into a simple story that stakeholders can follow.
- It shows whether the page moved because of a few standout queries or broad-based movement.

How to read it:
- Bars above zero added clicks.
- Bars below zero reduced clicks.
- The final bar shows the page's total net click change across all queries.

In [ ]:
# Query contribution waterfall for the page with the biggest query swing
TOP_WATERFALL_QUERIES = 12

# Change this page path if you want to inspect a different page story.
waterfall_focus_page = (
    query_summary.assign(abs_net_query_click_delta=query_summary["net_query_click_delta"].abs())
    .sort_values("abs_net_query_click_delta", ascending=False)
    .iloc[0]["page_path"]
)

waterfall_queries = (
    df_query_changes.loc[df_query_changes["page_path"] == waterfall_focus_page]
    .sort_values("click_delta", ascending=False)
    .copy()
)
if waterfall_queries.empty:
    raise ValueError(f"No query rows found for {waterfall_focus_page}")

top_positive = waterfall_queries.head(TOP_WATERFALL_QUERIES // 2)
top_negative = waterfall_queries.tail(TOP_WATERFALL_QUERIES - len(top_positive))
waterfall_focus = (
    pd.concat([top_positive, top_negative])
    .drop_duplicates(subset=["query"])
    .sort_values("click_delta", ascending=False)
    .copy()
)

other_mask = ~waterfall_queries.index.isin(waterfall_focus.index)
other_click_delta = waterfall_queries.loc[other_mask, "click_delta"].sum()

plot_rows = waterfall_focus[
    ["query", "query_state", "click_delta", "recent_clicks", "prior_clicks", "recent_impressions", "prior_impressions"]
].copy()
if other_mask.any() and other_click_delta != 0:
    plot_rows.loc[len(plot_rows)] = {
        "query": "All other queries",
        "query_state": "mixed",
        "click_delta": other_click_delta,
        "recent_clicks": waterfall_queries.loc[other_mask, "recent_clicks"].sum(),
        "prior_clicks": waterfall_queries.loc[other_mask, "prior_clicks"].sum(),
        "recent_impressions": waterfall_queries.loc[other_mask, "recent_impressions"].sum(),
        "prior_impressions": waterfall_queries.loc[other_mask, "prior_impressions"].sum(),
    }

plot_rows["label"] = plot_rows["query"].where(
    plot_rows["query"].str.len() <= 36,
    plot_rows["query"].str.slice(0, 36) + "...",
)
net_click_delta = int(waterfall_queries["click_delta"].sum())

waterfall_customdata = [
    [
        row.query,
        row.query_state.replace("_", " "),
        int(row.recent_clicks),
        int(row.prior_clicks),
        int(row.recent_impressions),
        int(row.prior_impressions),
    ]
    for row in plot_rows.itertuples()
]
waterfall_customdata.append(
    [
        "Net page change",
        "page total",
        int(waterfall_queries["recent_clicks"].sum()),
        int(waterfall_queries["prior_clicks"].sum()),
        int(waterfall_queries["recent_impressions"].sum()),
        int(waterfall_queries["prior_impressions"].sum()),
    ]
)

waterfall_text = [f"{value:+,.0f}" for value in plot_rows["click_delta"]] + [f"{net_click_delta:+,.0f}"]

fig_query_waterfall = go.Figure(
    go.Waterfall(
        measure=["relative"] * len(plot_rows) + ["total"],
        x=plot_rows["label"].tolist() + ["Net page change"],
        y=plot_rows["click_delta"].tolist() + [0],
        customdata=waterfall_customdata,
        connector={"line": {"color": "rgba(30, 60, 90, 0.30)"}},
        increasing={"marker": {"color": "#2E8540"}},
        decreasing={"marker": {"color": "#C62828"}},
        totals={"marker": {"color": "#1D5B8F"}},
        text=waterfall_text,
        textposition="outside",
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Contribution: %{text}<br>"
            "Query state: %{customdata[1]}<br>"
            "Recent clicks: %{customdata[2]}<br>"
            "Prior clicks: %{customdata[3]}<br>"
            "Recent impressions: %{customdata[4]}<br>"
            "Prior impressions: %{customdata[5]}<extra></extra>"
        ),
    )
)
fig_query_waterfall.update_layout(
    template="lifeline",
    title=(
        "Query contribution waterfall"
        "<br><span style='font-size:0.8em; font-weight:400;'>"
        f"{waterfall_focus_page} | Top positive and negative query contributions"
        "</span>"
    ),
    yaxis_title="Click delta",
    xaxis_title="Query",
    showlegend=False,
    margin=dict(l=60, r=40, t=80, b=120),  # Increase bottom margin to avoid cutting off text
    height=520,  # Optionally slightly increase height for more space for x-axis labels
)
lifeline_theme.add_lifeline_logo(fig_query_waterfall)
fig_query_waterfall.show()